In [2]:
import pandas as pd
import re

# =====================================================
# Match unique case IDs between Metasheet_Analysis.xlsx
# and AUR_ClinicalData_20260427.NO.DATES.bloodID.withBCRv2.xlsx
# =====================================================

metasheet_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis.xlsx"

clinical_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/AUR_ClinicalData_20260427.NO.DATES.bloodID.withBCRv2.xlsx"


# -----------------------------
# Helper function: extract case ID
# Example:
# AUP-ANU1-NB1 -> AUP-ANU1
# AUP-AQ0Z     -> AUP-AQ0Z
# -----------------------------
def extract_case_id(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    m = re.search(r"(AUP-[A-Za-z0-9]+)", x)

    return m.group(1) if m else pd.NA


# -----------------------------
# Read Metasheet_Analysis.xlsx
# -----------------------------
meta = pd.read_excel(metasheet_file)

meta["Case_ID"] = meta["Collaborator_sample_id"].apply(extract_case_id)

meta_cases = set(meta["Case_ID"].dropna().unique())


# -----------------------------
# Read AUR clinical data file
# -----------------------------
clinical = pd.read_excel(clinical_file)

# In case duplicate Patient_BCR_Barcode columns exist,
# pandas may rename them as Patient_BCR_Barcode.1, etc.
barcode_cols = [
    c for c in clinical.columns
    if str(c).startswith("Patient_BCR_Barcode")
]

clinical_case_id_df = clinical[barcode_cols].apply(
    lambda col: col.apply(extract_case_id)
)

clinical["Case_ID"] = clinical_case_id_df.bfill(axis=1).iloc[:, 0]

clinical_cases = set(clinical["Case_ID"].dropna().unique())


# -----------------------------
# Match unique cases
# -----------------------------
matched_cases = meta_cases.intersection(clinical_cases)

meta_not_in_clinical = meta_cases - clinical_cases
clinical_not_in_meta = clinical_cases - meta_cases


# -----------------------------
# Print summary
# -----------------------------
print("================ CASE MATCH SUMMARY ================")
print(f"Unique cases in Metasheet_Analysis.xlsx: {len(meta_cases)}")
print(f"Unique cases in AUR clinical data file: {len(clinical_cases)}")
print(f"Number of matched unique cases: {len(matched_cases)}")
print(f"Cases in Metasheet but not in AUR clinical data file: {len(meta_not_in_clinical)}")
print(f"Cases in AUR clinical data file but not in Metasheet: {len(clinical_not_in_meta)}")

================ CASE MATCH SUMMARY ================
Unique cases in Metasheet_Analysis.xlsx: 270
Unique cases in AUR clinical data file: 171
Number of matched unique cases: 164
Cases in Metasheet but not in AUR clinical data file: 106
Cases in AUR clinical data file but not in Metasheet: 7


In [3]:
# =====================================================
# Receptor status availability among matched cases
# =====================================================

print("\n================ RECEPTOR STATUS IN MATCHED CASES ================")

# Keep only rows from matched cases
clinical_matched = clinical[clinical["Case_ID"].isin(matched_cases)].copy()

# Clean receptor status values
def clean_status(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x == "" or x.lower() in ["nan", "na", "n/a", "none"]:
        return pd.NA
    return x

status_cols = ["ER_Status", "PR_Status", "HER2_Status"]

for col in status_cols:
    clinical_matched[col] = clinical_matched[col].apply(clean_status)

# One receptor-status value per unique case
# If a case has multiple rows, take the first non-missing value
def first_non_missing(series):
    vals = series.dropna().unique()
    return vals[0] if len(vals) > 0 else pd.NA

case_receptor_status = (
    clinical_matched
    .groupby("Case_ID", as_index=False)[status_cols]
    .agg(first_non_missing)
)

# Count availability
n_matched_cases = len(matched_cases)

n_er = case_receptor_status["ER_Status"].notna().sum()
n_pr = case_receptor_status["PR_Status"].notna().sum()
n_her2 = case_receptor_status["HER2_Status"].notna().sum()

n_all_3 = (
    case_receptor_status["ER_Status"].notna()
    & case_receptor_status["PR_Status"].notna()
    & case_receptor_status["HER2_Status"].notna()
).sum()

print(f"Total matched unique cases: {n_matched_cases}")
print(f"Matched cases with ER_Status available: {n_er}")
print(f"Matched cases with PR_Status available: {n_pr}")
print(f"Matched cases with HER2_Status available: {n_her2}")
print(f"Matched cases with all 3 receptor statuses available: {n_all_3}")

print("\nER_Status counts among matched cases:")
print(case_receptor_status["ER_Status"].fillna("Missing/blank").value_counts())

print("\nPR_Status counts among matched cases:")
print(case_receptor_status["PR_Status"].fillna("Missing/blank").value_counts())

print("\nHER2_Status counts among matched cases:")
print(case_receptor_status["HER2_Status"].fillna("Missing/blank").value_counts())


================ RECEPTOR STATUS IN MATCHED CASES ================
Total matched unique cases: 164
Matched cases with ER_Status available: 164
Matched cases with PR_Status available: 164
Matched cases with HER2_Status available: 164
Matched cases with all 3 receptor statuses available: 164

ER_Status counts among matched cases:
Positive    136
Negative     26
Unknown       2
Name: ER_Status, dtype: int64

PR_Status counts among matched cases:
Positive    116
Negative     45
Unknown       3
Name: PR_Status, dtype: int64

HER2_Status counts among matched cases:
Negative    135
Positive     23
Unknown       6
Name: HER2_Status, dtype: int64


In [4]:
# =====================================================
# Histological Type availability among matched cases
# =====================================================

print("\n================ HISTOLOGICAL TYPE IN MATCHED CASES ================")

# Keep only rows from matched cases
clinical_matched = clinical[clinical["Case_ID"].isin(matched_cases)].copy()

# Clean Histologic.type values
def clean_histology(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x == "" or x.lower() in ["nan", "na", "n/a", "none"]:
        return pd.NA
    return x

clinical_matched["Histologic.type"] = clinical_matched["Histologic.type"].apply(clean_histology)

# One histology value per unique case
# If a case has multiple rows, take the first non-missing value
def first_non_missing(series):
    vals = series.dropna().unique()
    return vals[0] if len(vals) > 0 else pd.NA

case_histology = (
    clinical_matched
    .groupby("Case_ID", as_index=False)["Histologic.type"]
    .agg(first_non_missing)
)

# Make sure all matched cases are represented
all_matched_cases_df = pd.DataFrame({"Case_ID": sorted(matched_cases)})

case_histology_full = all_matched_cases_df.merge(
    case_histology,
    on="Case_ID",
    how="left"
)

# Count availability
n_matched_cases = len(case_histology_full)
n_hist_available = case_histology_full["Histologic.type"].notna().sum()
n_hist_missing = case_histology_full["Histologic.type"].isna().sum()

print(f"Total matched unique cases: {n_matched_cases}")
print(f"Matched cases with Histologic.type available: {n_hist_available}")
print(f"Matched cases with Histologic.type missing: {n_hist_missing}")

print("\nUnique Histologic.type values among matched cases:")
print(case_histology_full["Histologic.type"].dropna().nunique())

print("\nHistologic.type counts among matched cases:")
print(
    case_histology_full["Histologic.type"]
    .fillna("Missing/blank")
    .value_counts()
)


================ HISTOLOGICAL TYPE IN MATCHED CASES ================
Total matched unique cases: 164
Matched cases with Histologic.type available: 159
Matched cases with Histologic.type missing: 5

Unique Histologic.type values among matched cases:
3

Histologic.type counts among matched cases:
Ductal                      133
Lobular                      17
Mixed Ductal and Lobular      9
Missing/blank                 5
Name: Histologic.type, dtype: int64


In [13]:
# =====================================================
# Fill Metasheet_Analysis.xlsx using AUR clinical data
# and print histology / molecular subtype distribution
# =====================================================

print("\n================ FILL METASHEET FROM AUR CLINICAL DATA ================")

# -----------------------------
# Columns to fill
# -----------------------------
target_source_map = {
    "Clinical_ID": "Clinical_ID",
    "Clinical_ID_2": "Clinical_ID_2",
    "Histological_subtype": "Histologic.type",
    "ER_Status": "ER_Status",
    "PR_Status": "PR_Status",
    "HER2_Status": "HER2_Status"
}

# -----------------------------
# Clean helper
# -----------------------------
def clean_value(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x == "" or x.lower() in ["nan", "na", "n/a", "none"]:
        return pd.NA
    return x


# -----------------------------
# Collapse clinical data to one row per case
# -----------------------------
clinical_for_fill = clinical.copy()

for col in target_source_map.values():
    clinical_for_fill[col] = clinical_for_fill[col].apply(clean_value)

# For Clinical_ID and Clinical_ID_2:
# If multiple IDs exist for one case, keep unique values separated by "; "
def join_unique_values(series):
    vals = []
    for x in series.dropna():
        if x not in vals:
            vals.append(x)
    if len(vals) == 0:
        return pd.NA
    return "; ".join(vals)

# For histology/receptor status:
# If one unique value exists, use it.
# If multiple different values exist, leave empty and flag for review.
def single_unique_or_blank(series):
    vals = []
    for x in series.dropna():
        if x not in vals:
            vals.append(x)
    if len(vals) == 0:
        return pd.NA
    if len(vals) == 1:
        return vals[0]
    return pd.NA

case_annotation = (
    clinical_for_fill
    .groupby("Case_ID", as_index=False)
    .agg({
        "Clinical_ID": join_unique_values,
        "Clinical_ID_2": join_unique_values,
        "Histologic.type": single_unique_or_blank,
        "ER_Status": single_unique_or_blank,
        "PR_Status": single_unique_or_blank,
        "HER2_Status": single_unique_or_blank
    })
)

# Rename source clinical columns to metasheet column names
case_annotation = case_annotation.rename(columns={
    "Histologic.type": "Histological_subtype"
})


# -----------------------------
# Prepare metasheet for filling
# -----------------------------
meta_filled = meta.copy()

# Make sure Case_ID exists in metasheet
meta_filled["Case_ID"] = meta_filled["Collaborator_sample_id"].apply(extract_case_id)

# Merge clinical annotation into metasheet
meta_filled = meta_filled.merge(
    case_annotation,
    on="Case_ID",
    how="left",
    suffixes=("", "_from_AUR")
)

# Fill only empty cells in the original metasheet columns
for target_col in target_source_map.keys():
    source_col = f"{target_col}_from_AUR"

    if source_col in meta_filled.columns:
        meta_filled[target_col] = meta_filled[target_col].apply(clean_value)

        meta_filled[target_col] = meta_filled[target_col].combine_first(
            meta_filled[source_col]
        )

        meta_filled = meta_filled.drop(columns=[source_col])


# -----------------------------
# Derive Molecular_subtype
# only if ER, PR, and HER2 are available
# -----------------------------
# =====================================================
# Derive 4-class Molecular_subtype
# HR+/HER2+, HR+/HER2-, HR-/HER2+, HR-/HER2-
# =====================================================

def normalize_receptor_status(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip().lower()

    if x in ["positive", "pos", "+"]:
        return "Positive"

    if x in ["negative", "neg", "-"]:
        return "Negative"

    # Treat Unknown/Equivocal/NA as missing
    if x in ["unknown", "equivocal", "indeterminate", "nan", "na", "n/a", "none", ""]:
        return pd.NA

    return pd.NA


def derive_molecular_subtype(row):
    er = normalize_receptor_status(row["ER_Status"])
    pr = normalize_receptor_status(row["PR_Status"])
    her2 = normalize_receptor_status(row["HER2_Status"])

    # Need HER2 to define HER2+ or HER2-
    if pd.isna(her2):
        return pd.NA

    # Need both ER and PR to define HR status safely
    if pd.isna(er) or pd.isna(pr):
        return pd.NA

    # HR status
    if er == "Positive" or pr == "Positive":
        hr = "HR+"
    elif er == "Negative" and pr == "Negative":
        hr = "HR-"
    else:
        return pd.NA

    # HER2 status
    if her2 == "Positive":
        her2_group = "HER2+"
    elif her2 == "Negative":
        her2_group = "HER2-"
    else:
        return pd.NA

    return f"{hr}/{her2_group}"

# Only fill empty Molecular_subtype cells
meta_filled["Molecular_subtype"] = meta_filled["Molecular_subtype"].apply(clean_value)

derived_molecular_subtype = meta_filled.apply(derive_molecular_subtype, axis=1)

meta_filled["Molecular_subtype"] = meta_filled["Molecular_subtype"].combine_first(
    derived_molecular_subtype
)

# =====================================================
# QC: Identify conflicting Histologic.type and Molecular_subtype
# among matched cases
# =====================================================

print("\n================ CONFLICT CHECK AMONG MATCHED CASES ================")

# Keep only matched clinical rows
clinical_matched_qc = clinical_for_fill[
    clinical_for_fill["Case_ID"].isin(matched_cases)
].copy()


# -----------------------------
# Conflicting Histologic.type
# -----------------------------
hist_conflict_check = (
    clinical_matched_qc
    .dropna(subset=["Histologic.type"])
    .groupby("Case_ID")["Histologic.type"]
    .agg(lambda x: sorted(set(x.dropna())))
    .reset_index()
)

hist_conflict_check["n_unique_histology"] = hist_conflict_check["Histologic.type"].apply(len)

conflicting_histology_cases = hist_conflict_check[
    hist_conflict_check["n_unique_histology"] > 1
].copy()

print(
    f"Number of matched cases with conflicting Histologic.type values: "
    f"{len(conflicting_histology_cases)}"
)

if len(conflicting_histology_cases) > 0:
    print("\nConflicting Histologic.type cases:")
    print(conflicting_histology_cases.to_string(index=False))
else:
    print("No conflicting Histologic.type cases found among matched cases.")


# -----------------------------
# Conflicting Molecular_subtype
# derived from ER_Status, PR_Status, HER2_Status
# -----------------------------
def derive_molecular_subtype_from_values(er, pr, her2):
    er = normalize_receptor_status(er)
    pr = normalize_receptor_status(pr)
    her2 = normalize_receptor_status(her2)

    if pd.isna(er) or pd.isna(pr) or pd.isna(her2):
        return pd.NA

    if er == "Positive" or pr == "Positive":
        hr = "HR+"
    elif er == "Negative" and pr == "Negative":
        hr = "HR-"
    else:
        return pd.NA

    if her2 == "Positive":
        her2_group = "HER2+"
    elif her2 == "Negative":
        her2_group = "HER2-"
    else:
        return pd.NA

    return f"{hr}/{her2_group}"


clinical_matched_qc["Derived_Molecular_subtype"] = clinical_matched_qc.apply(
    lambda row: derive_molecular_subtype_from_values(
        row["ER_Status"],
        row["PR_Status"],
        row["HER2_Status"]
    ),
    axis=1
)

molecular_conflict_check = (
    clinical_matched_qc
    .dropna(subset=["Derived_Molecular_subtype"])
    .groupby("Case_ID")["Derived_Molecular_subtype"]
    .agg(lambda x: sorted(set(x.dropna())))
    .reset_index()
)

molecular_conflict_check["n_unique_molecular_subtype"] = molecular_conflict_check[
    "Derived_Molecular_subtype"
].apply(len)

conflicting_molecular_cases = molecular_conflict_check[
    molecular_conflict_check["n_unique_molecular_subtype"] > 1
].copy()

print(
    f"\nNumber of matched cases with conflicting Molecular_subtype values: "
    f"{len(conflicting_molecular_cases)}"
)

if len(conflicting_molecular_cases) > 0:
    print("\nConflicting Molecular_subtype cases:")
    print(conflicting_molecular_cases.to_string(index=False))
else:
    print("No conflicting Molecular_subtype cases found among matched cases.")


# =====================================================
# Print matched-case-only distributions after filling
# =====================================================

print("\n================ MATCHED CASE DISTRIBUTION AFTER FILLING ================")

# Keep only samples from matched cases
matched_sample_meta = meta_filled[
    meta_filled["Case_ID"].isin(matched_cases)
].copy()

# Clean columns before counting
for col in ["Histological_subtype", "Molecular_subtype"]:
    matched_sample_meta[col] = matched_sample_meta[col].apply(clean_value)

# One row per matched case
matched_case_meta = (
    matched_sample_meta
    .dropna(subset=["Case_ID"])
    .drop_duplicates(subset=["Case_ID"])
    .copy()
)

print(f"Total matched unique cases: {matched_case_meta['Case_ID'].nunique()}")


# -----------------------------
# Histologic.type availability and counts
# -----------------------------
n_hist_available = matched_case_meta["Histological_subtype"].notna().sum()
n_hist_missing = matched_case_meta["Histological_subtype"].isna().sum()

print(f"\nMatched cases with Histologic.type available: {n_hist_available}")
print(f"Matched cases with Histologic.type missing: {n_hist_missing}")

print("\nHistologic.type counts among matched cases:")
print(
    matched_case_meta["Histological_subtype"]
    .fillna("Missing/blank")
    .value_counts()
)


# =====================================================
# Case-level distribution among matched cases only
# =====================================================

print("\n================ CASE-LEVEL DISTRIBUTION AMONG MATCHED CASES ================")

print("\nHistological_subtype distribution at matched case level:")
print(
    matched_case_meta["Histological_subtype"]
    .fillna("Missing/blank")
    .value_counts()
)

print("\nMolecular_subtype distribution at matched case level:")
print(
    matched_case_meta["Molecular_subtype"]
    .fillna("Missing/blank")
    .value_counts()
)


# =====================================================
# Sample-level distribution among matched cases only
# =====================================================

print("\n================ SAMPLE-LEVEL DISTRIBUTION AMONG MATCHED CASES ================")

print("\nHistological_subtype distribution at matched sample level:")
print(
    matched_sample_meta["Histological_subtype"]
    .fillna("Missing/blank")
    .value_counts()
)

print("\nMolecular_subtype distribution at matched sample level:")
print(
    matched_sample_meta["Molecular_subtype"]
    .fillna("Missing/blank")
    .value_counts()
)
# =====================================================
# Save filled metasheet
# =====================================================

output_file = "/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/Metasheet_Analysis_filled_from_AUR_ClinicalData.xlsx"

meta_filled.to_excel(output_file, index=False)

print(f"\nSaved filled metasheet to:\n{output_file}")


================ FILL METASHEET FROM AUR CLINICAL DATA ================

================ CONFLICT CHECK AMONG MATCHED CASES ================
Number of matched cases with conflicting Histologic.type values: 4

Conflicting Histologic.type cases:
 Case_ID                    Histologic.type  n_unique_histology
AUP-ANF8                  [Ductal, Lobular]                   2
AUP-ANU1 [Ductal, Mixed Ductal and Lobular]                   2
AUP-AOJF                  [Ductal, Lobular]                   2
AUP-APMG [Ductal, Mixed Ductal and Lobular]                   2

Number of matched cases with conflicting Molecular_subtype values: 44

Conflicting Molecular_subtype cases:
 Case_ID         Derived_Molecular_subtype  n_unique_molecular_subtype
AUP-AL6N            [HR-/HER2+, HR-/HER2-]                           2
AUP-AL9X            [HR+/HER2-, HR-/HER2-]                           2
AUP-ALTT            [HR+/HER2-, HR-/HER2-]                           2
AUP-ALW9            [HR+/HER2+, HR+/HER2-